In [1]:
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from shapely.geometry import Point
import pygris
pd.set_option("display.max_rows", None)
import requests

In [21]:
# Elizabeth's API Key from census.gov
usr_key = '972d7153c997cbab90a0069f502742433ea382e1' # raw key only, no &key= prefix

# Tables wanted
TABLES = [
    "B18102",
    "B18103",
    "B18104",
    "B18105",
    "B18106",
    "B18107",
  #  "B18108",
    "B27001"
]

ACS_YEAR = 2024

In [3]:
#define project parameters
ADDRESS = "12007 Courthouse Cir, New Kent, VA 23124"
BUFFER_MILES = 25
PROJECTED_CRS = "EPSG:2284"  # Virginia South State Plane (feet)

In [57]:
### NOTE !!!! my census tracts shapefile has FIPS codes. 
### Check yours or use pygris to get lines. Change to GEOID as needed.
# I rename FIPS to GEOID before merging with census data.

In [4]:
#setup geocoding address
geolocator = Nominatim(user_agent="census_buffer_analysis")
location = geolocator.geocode(ADDRESS)

point = Point(location.longitude, location.latitude)

site = gpd.GeoDataFrame(
    {"Name": ["Project Site"]},
    geometry=[point],
    crs="EPSG:4326"
)

In [5]:
#create 25 mi buffer
site_proj = site.to_crs(PROJECTED_CRS)

buffer_ft = BUFFER_MILES * 5280

radius = site_proj.copy()
radius["geometry"] = radius.buffer(buffer_ft)

In [6]:
#read tracts
va_tracts_full = gpd.read_file(r'C:\Users\ElizabethGreenwell\OneDrive - PlanRVA\BasicData\Boundaries\CensusTractsVA.shp', engine='fiona')

va_tracts_full = va_tracts_full.to_crs(PROJECTED_CRS)

In [7]:
#select tracts with buffer
tracts = gpd.overlay(
    va_tracts_full,
    radius,
    how="intersection"
)

In [8]:
full_area = (
    va_tracts_full[["FIPS", "geometry"]]
    .copy()
)

full_area["tract_area"] = full_area.area

tracts["intersect_area"] = tracts.area

tracts = tracts.merge(
    full_area[["FIPS", "tract_area"]],
    on="FIPS",
    how="left"
)

tracts["weight"] = (
    tracts["intersect_area"] /
    tracts["tract_area"]
)

tracts["weight"] = tracts["weight"].clip(upper=1)

tracts.rename(columns={"FIPS": "GEOID"}, inplace=True)

In [22]:
#make a function to get variables
def get_table_variables(table_id, year):

    metadata_url = (
        f"https://api.census.gov/data/{year}/acs/acs5/variables.json"
    )

    r = requests.get(metadata_url)
    if r.status_code != 200:
        raise RuntimeError(f"Census metadata request failed: {r.status_code}\n{r.text[:200]}")

    try:
        meta = r.json()
    except Exception as e:
        raise RuntimeError(f"Census metadata returned non-JSON.\nURL: {metadata_url}\nResponse: {r.text[:300]}") from e

    vars_dict = meta["variables"]

    variables = [
        v
        for v in vars_dict.keys()
        if v.startswith(table_id)
        and v.endswith("E")
    ]

    return sorted(variables)

In [23]:
#download table data
all_results = []

for table in TABLES:

    print(f"Processing {table}")

    vars_list = get_table_variables(
        table,
        ACS_YEAR
    )

    print(f"{len(vars_list)} variables found")

    chunks = [
        vars_list[i:i+45]
        for i in range(0, len(vars_list), 45)
    ]

    merged = None

    for chunk in chunks:

        var_string = ",".join(chunk)

        url = (
            f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
            f"?get={var_string}"
            f"&for=tract:*"
            f"&in=state:51"
            f"&key={usr_key}"
        )

        r = requests.get(url)

        data = r.json()

        df = pd.DataFrame(
            data[1:],
            columns=data[0]
        )

        df["GEOID"] = (
            df["state"]
            + df["county"]
            + df["tract"]
        )

        keep_cols = ["GEOID"] + chunk

        df = df[keep_cols]

        if merged is None:
            merged = df
        else:
            merged = merged.merge(
                df,
                on="GEOID",
                how="left"
            )

    # Convert to numeric
    for c in merged.columns:
        if c != "GEOID":
            merged[c] = pd.to_numeric(
                merged[c],
                errors="coerce"
            )

    # Join to clipped tracts
    analysis = tracts.merge(
        merged,
        on="GEOID",
        how="left"
    )

    # Area-weight all estimate fields
    estimate_fields = [
        c for c in merged.columns
        if c != "GEOID"
    ]

    results = []

    for field in estimate_fields:

        estimate = (
            analysis[field]
            * analysis["weight"]
        ).sum()

        results.append({
            "Table": table,
            "Variable": field,
            "Estimate_25_Mile_Radius": estimate
        })

    all_results.append(
        pd.DataFrame(results)
    )

Processing B18102
39 variables found
Processing B18103
39 variables found
Processing B18104
33 variables found
Processing B18105
33 variables found
Processing B18106
33 variables found
Processing B18107
27 variables found
Processing B27001
57 variables found


In [24]:
#export results
results_df = pd.concat(
    all_results,
    ignore_index=True
)

results_df.to_csv(
    "ACS_25Mile_Radius_Estimates.csv",
    index=False
)

print(results_df.head())
print("Finished.")

    Table     Variable  Estimate_25_Mile_Radius
0  B18102  B18102_001E            404288.351426
1  B18102  B18102_002E            193317.729137
2  B18102  B18102_003E             11524.285756
3  B18102  B18102_004E                60.270290
4  B18102  B18102_005E             11464.015466
Finished.
